# 02 — Direct edge-based QUBO validation

The direct QUBO formulation is the closest quantum-optimization representation of the constrained OEC model considered in the paper. Each binary semantic variable retains the meaning of one service using one admissible edge of the time-expanded graph, while source, destination and flow-conservation equalities are transferred to the objective through quadratic penalties. Shared resource inequalities require additional slack bits, and this second contribution to the logical size is kept explicit because it becomes relevant in the later resource-scaling analysis.

This notebook performs the first equivalence test between the classical constrained formulation and the manually constructed QUBO. The comparison is deliberately carried out on the same canonical instance C1 introduced in the previous notebook, so that a disagreement can be attributed to the encoding itself rather than to a change of physical scenario.

In [ ]:
from __future__ import annotations

import pandas as pd

from oec_qaoa import canonical_c1
from oec_qaoa.edge_qubo import build_edge_qubo, solve_qubo_exact
from oec_qaoa.ilp import solve_reference_ilp

instance = canonical_c1()
reference = solve_reference_ilp(instance)
qubo = build_edge_qubo(instance)

print(f"Reference objective: {reference.objective:.3f}")
print(f"QUBO penalty: {qubo.penalty:.3f}")

## Service-specific preprocessing

The complete time-expanded graph contains transitions that are physically valid for the constellation but cannot belong to any source-to-destination path of a given service. Before introducing binary variables, the implementation therefore removes edges that are either unreachable from the acquisition state or unable to reach the processed ground-state destination within the service deadline.

This preprocessing does not select a route and does not approximate the optimization problem. It only restricts each service to the portion of the graph that can participate in at least one admissible path, which is equivalent to removing decision variables that would be fixed to zero in every feasible solution.

In [ ]:
summary = pd.DataFrame(
    {
        "quantity": [
            "semantic edge variables",
            "slack variables",
            "total logical variables",
            "quadratic couplings",
            "penalized equalities",
        ],
        "value": [
            qubo.semantic_count,
            len(qubo.variable_names) - qubo.semantic_count,
            len(qubo.variable_names),
            len(qubo.quadratic),
            len(qubo.constraints),
        ],
    }
)
summary

## Penalty construction

For C1 the unconstrained linear objective is non-negative and the sum of its positive coefficients is (7), corresponding to the two possible ground-processing contributions. The implementation selects a common penalty coefficient one unit larger than this objective span,

[
\lambda = 8,
]

so that any integer constraint violation of magnitude at least one costs more than the complete variation of the unpenalized objective. This is a sufficient bound for the present integer-scaled formulation, not a claim that the same numerical value should be used for larger instances.

Resource inequalities are converted to equalities only when they can actually bind over the preprocessed binary-variable box. The slack variable (s_r) associated with a capacity (C_r) is encoded in powers of two and enters

[
\sum_j a_{rj} y_j + s_r = C_r.
]

Algebraically identical inequalities are represented once. In C1 the satellite processing-capacity limit and the corresponding energy limit happen to impose the same binary inequality, so keeping both would only duplicate a penalty without changing the feasible set.

In [ ]:
constraint_rows = []
for constraint in qubo.constraints:
    constraint_rows.append(
        {
            "constraint": constraint.name,
            "terms": len(constraint.coefficients),
            "rhs": constraint.rhs,
            "type": constraint.name.split(":")[0],
        }
    )

constraints = pd.DataFrame(constraint_rows)
constraints.groupby("type").agg(
    count=("constraint", "count"),
    mean_terms=("terms", "mean"),
    max_terms=("terms", "max"),
)

In [ ]:
slack_rows = []
for label, indices in qubo.slack_variables.items():
    slack_rows.append(
        {
            "constraint": label,
            "number_of_slack_bits": len(indices),
            "variables": ", ".join(
                qubo.variable_names[index] for index in indices
            ),
        }
    )

pd.DataFrame(slack_rows)

The direct C1 QUBO therefore contains the complete service-edge semantics together with the slack bits required by the two active shared-resource inequalities. The purpose of this count is already broader than the validation instance: later notebooks will repeat the same accounting as the constellation, time horizon and number of services increase, allowing the model reduction to be discussed in terms of logical variables and interaction density rather than only objective quality.

## Exact QUBO solution

The QUBO is minimized here with an exact classical verification routine. Quadratic products are linearized with auxiliary continuous variables and the standard McCormick envelope, while the original QUBO variables remain binary. This solver is used only to certify the minimum of the binary quadratic model; it is not part of the proposed quantum method and will not be used as a performance baseline for QAOA.

The relevant equivalence test is stronger than equality of the scalar objective. After minimization, the semantic part of the QUBO bit string is decoded back into service-edge decisions and compared directly with the constrained OEC solution.

In [ ]:
qubo_solution = solve_qubo_exact(qubo)

print(f"QUBO solver success: {qubo_solution.success}")
print(f"QUBO minimum energy: {qubo_solution.energy:.3f}")
print(qubo_solution.message)

In [ ]:
comparison = pd.DataFrame(
    [
        {
            "service": service.name,
            "reference_edges": reference.selected_edges[service.name],
            "qubo_edges": qubo_solution.selected_edges[service.name],
            "same_decoded_solution": (
                reference.selected_edges[service.name]
                == qubo_solution.selected_edges[service.name]
            ),
        }
        for service in instance.services
    ]
)
comparison

## Equivalence gate

For C1 the QUBO is accepted only if three conditions hold simultaneously: the exact QUBO energy equals the constrained optimum, the semantic variables decode to the same physical service paths, and the known processing-placement structure is preserved. The test is intentionally strict because the same formulation will later be applied to less transparent synthetic instances, where visual inspection of the optimum is no longer sufficient.

In [ ]:
assert reference.success
assert qubo_solution.success
assert abs(qubo_solution.energy - reference.objective) < 1e-9
assert qubo_solution.selected_edges == reference.selected_edges

assert qubo_solution.selected_edges["h0"] == (
    "tx_S0_S1_t0_p0",
    "st_S1_t0_p0",
    "st_S1_t1_p1",
    "st_G_t2_p1",
    "proc_S1_t1",
    "tx_S1_G_t2_p1",
)

assert qubo_solution.selected_edges["h1"] == (
    "tx_S2_S1_t0_p0",
    "st_S1_t0_p0",
    "st_S1_t1_p0",
    "st_G_t2_p0",
    "tx_S1_G_t2_p0",
    "proc_G_t3",
)

print("C1 edge-QUBO equivalence: PASS")

## Interpretation and next validation step

The C1 result establishes only an encoding property: for this controlled instance, the manually derived QUBO preserves the optimum and the decoded physical decisions of the original constrained model. It does not yet provide evidence about the performance of QAOA, nor does it show that the direct encoding remains manageable as the OEC graph grows.

The next implementation stage will therefore introduce a second validation instance with genuine alternative routes and then move to the path-based reduction. This sequence is useful because it separates three questions that should remain distinct in the final paper: whether the QUBO is faithful to the OEC problem, how much approximation is introduced by classical path restriction, and how accurately a variational quantum solver subsequently optimizes the reduced binary model.